In [1]:
# CRITICAL: restrict to one GPU before any CUDA initialization
# (prevents the DataParallel memory-gather crash from earlier experiments)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Restricted to GPU 0")

Restricted to GPU 0


# Multilingual Health QA — AfriTeVa V2 Fine-Tuning
## Experiment 5: Africa-Centric Seq2Seq Model

**Author:** Samuel Mwania
**Model:** castorini/afriteva_v2_base (659M params, T5 pretrained on African languages)

### Why AfriTeVa V2 over mT5
Empirical testing showed AfriTeVa V2 handles all 8 language configs with minimal
tokenizer loss (Akan 2.0 UNK/1000 tokens, all others <1.0). Published results show
it surpasses mT5-base by up to 10 points on African-language generation because it
saw far more African-language data during pretraining.

### Training strategy
- 8 epochs with per-epoch validation and best-checkpoint retention (load_best_model_at_end)
- This finds the optimal stopping point empirically rather than guessing epoch count
- Stable config proven in earlier experiments: fp32 + Adafactor + gradient checkpointing
- Model saved persistently to avoid session-cycling data loss

In [2]:
!pip install -q -U transformers==4.44.2 datasets==2.21.0 sentencepiece==0.2.0 accelerate==0.34.2
print("Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires goog

In [3]:
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seed:", SEED)

2026-06-05 08:16:36.791692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780647397.010198      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780647397.068440      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780647397.572544      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780647397.572620      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780647397.572629      24 computation_placer.cc:177] computation placer alr

PyTorch: 2.10.0+cu128
CUDA available: True
Visible GPUs: 1
GPU: Tesla T4
Seed: 42


In [4]:
# Load and clean data (same conservative cleaning as all prior experiments)
DATA_DIR = '/kaggle/input/datasets/samuelmwania1/multilingual-health-qa-data'
train_df = pd.read_csv(f'{DATA_DIR}/Train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/Val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/Test.csv')

def clean(df):
    before = len(df)
    df = df[df['input'].str.split().str.len() > 0].copy()
    df = df.drop_duplicates(subset=['input', 'output'], keep='first').reset_index(drop=True)
    print(f"  {before} -> {len(df)} (removed {before-len(df)})")
    return df

print("Cleaning train:"); train_df = clean(train_df)
print("Cleaning val:");   val_df = clean(val_df)
print(f"\nFinal: {len(train_df):,} train | {len(val_df):,} val | {len(test_df):,} test")

Cleaning train:
  29815 -> 29538 (removed 277)
Cleaning val:
  6686 -> 6673 (removed 13)

Final: 29,538 train | 6,673 val | 2,618 test


In [5]:
# Language-aware prompts (the key fix; same as Experiment 2)
SUBSET_TO_LANG = {
    'Aka_Gha': 'Akan (Ghana)', 'Amh_Eth': 'Amharic (Ethiopia)',
    'Eng_Eth': 'English (Ethiopia)', 'Eng_Gha': 'English (Ghana)',
    'Eng_Ken': 'English (Kenya)', 'Eng_Uga': 'English (Uganda)',
    'Lug_Uga': 'Luganda (Uganda)', 'Swa_Ken': 'Swahili (Kenya)',
}

def build_prompt(question, subset):
    lang = SUBSET_TO_LANG.get(subset, 'the same language')
    return f"Answer this health question in {lang}: {str(question).strip()}"

for d in (train_df, val_df, test_df):
    d['prompt'] = d.apply(lambda r: build_prompt(r['input'], r['subset']), axis=1)

print("Prompts built. Example:")
print(train_df['prompt'].iloc[0][:120])

Prompts built. Example:
Answer this health question in Akan (Ghana): Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ w


In [6]:
# Load AfriTeVa V2
MODEL_NAME = 'castorini/afriteva_v2_base'
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Silence the tied-weights warning we saw during verification
model.config.tie_word_embeddings = False

device = torch.device('cuda')
model = model.to(device)
print(f"Loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {device}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

Loading castorini/afriteva_v2_base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.72G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Loaded: 428.9M params on cuda
Vocab size: 150,100


In [7]:
# Tokenize with EDA-justified lengths
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 256

def preprocess(examples):
    model_inputs = tokenizer(
        examples['prompt'], max_length=MAX_INPUT_LENGTH, truncation=True, padding=False
    )
    labels = tokenizer(
        text_target=examples['output'], max_length=MAX_TARGET_LENGTH, truncation=True, padding=False
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['prompt', 'output']])
val_ds   = Dataset.from_pandas(val_df[['prompt', 'output']])

print("Tokenizing train...")
train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
print("Tokenizing val...")
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
print(f"Done: {len(train_tok)} train, {len(val_tok)} val tokenized")

Tokenizing train...


Map:   0%|          | 0/29538 [00:00<?, ? examples/s]

Tokenizing val...


Map:   0%|          | 0/6673 [00:00<?, ? examples/s]

Done: 29538 train, 6673 val tokenized


In [8]:
# Stratified validation subset for fast per-epoch evaluation (125/language)
val_sub_df = (
    val_df.groupby('subset', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 125), random_state=SEED))
    .reset_index(drop=True)
)
val_sub_ds = Dataset.from_pandas(val_sub_df[['prompt', 'output']])
val_sub_tok = val_sub_ds.map(preprocess, batched=True, remove_columns=val_sub_ds.column_names)
print(f"Validation subset: {len(val_sub_tok)} ({val_sub_df['subset'].value_counts().min()}-{val_sub_df['subset'].value_counts().max()} per language)")

/tmp/ipykernel_24/446136567.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 125), random_state=SEED))


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Validation subset: 1000 (125-125 per language)


In [9]:
# Data collator: dynamic padding, label masking
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    label_pad_token_id=-100, pad_to_multiple_of=8
)

OUTPUT_DIR = '/kaggle/working/afriteva_exp5'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,                      # generous; best epoch auto-selected
    per_device_train_batch_size=4,           # safe for T4 in fp32
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,           # effective batch size 16
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    # Stable precision config proven in earlier experiments:
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='adafactor',
    save_safetensors=False,                  # avoids the non-contiguous-tensor save bug
    # Evaluation and checkpointing:
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,                      # keep best + last
    load_best_model_at_end=True,             # auto-keep lowest val loss epoch
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,
    logging_steps=100,
    report_to='none',
    seed=SEED,
)
print("Training args set")
print(f"Effective batch size: {4*4} | Epochs: 5 | Output: {OUTPUT_DIR}")

Training args set
Effective batch size: 16 | Epochs: 5 | Output: /kaggle/working/afriteva_exp5


In [10]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_sub_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
)
print("Trainer ready")
print(f"Steps/epoch: ~{len(train_tok)//16} | Total steps: ~{len(train_tok)//16*8}")

Trainer ready
Steps/epoch: ~1846 | Total steps: ~14768


In [11]:
print("Starting training...")
print(f"Examples: {len(train_tok):,} | Epochs: 5 | Effective batch: 16")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print(f"Final train loss: {train_result.training_loss:.4f}")

# Print the per-epoch eval history (this becomes your learning curve)
print("\nEpoch-by-epoch validation loss:")
for entry in trainer.state.log_history:
    if 'eval_loss' in entry:
        print(f"  Epoch {entry['epoch']:.0f}: eval_loss = {entry['eval_loss']:.4f}")

Starting training...
Examples: 29,538 | Epochs: 5 | Effective batch: 16


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
0,3.211700,2.788305
1,2.807900,2.524645
2,2.634000,2.401714
4,2.481300,2.322006



TRAINING COMPLETE
Final train loss: 2.9903

Epoch-by-epoch validation loss:
  Epoch 1: eval_loss = 2.7883
  Epoch 2: eval_loss = 2.5246
  Epoch 3: eval_loss = 2.4017
  Epoch 4: eval_loss = 2.3432
  Epoch 5: eval_loss = 2.3220


In [12]:
# Save the best model to working dir (persists as commit output)
FINAL_DIR = '/kaggle/working/afriteva_best_model'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Best model saved to {FINAL_DIR}")
import os
print("Saved files:", os.listdir(FINAL_DIR))

Best model saved to /kaggle/working/afriteva_best_model
Saved files: ['tokenizer_config.json', 'pytorch_model.bin', 'spiece.model', 'generation_config.json', 'config.json', 'special_tokens_map.json', 'tokenizer.json', 'training_args.bin']


In [13]:
# Generation with tuned decoding
model.config.use_cache = True
model.eval()

def generate_batch(prompts, batch_size=16, num_beams=4, max_new_tokens=256,
                   min_new_tokens=15, length_penalty=1.2, no_repeat_ngram_size=3):
    out = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=MAX_INPUT_LENGTH).to(device)
        with torch.no_grad():
            gen = model.generate(
                **inputs, num_beams=num_beams,
                max_new_tokens=max_new_tokens, min_new_tokens=min_new_tokens,
                length_penalty=length_penalty, no_repeat_ngram_size=no_repeat_ngram_size,
                early_stopping=True
            )
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size, len(prompts))}/{len(prompts)}")
    return out

print("Generating test answers...")
test_answers = generate_batch(test_df['prompt'].tolist())
print(f"Generated {len(test_answers)}")

Generating test answers...
  16/2618
  336/2618
  656/2618
  976/2618
  1296/2618
  1616/2618
  1936/2618
  2256/2618
  2576/2618
Generated 2618


In [14]:
def clean_generated(text):
    text = str(text).strip()
    text = re.sub(r'<extra_id_\d+>', '', text)
    text = re.sub(r'^This is a question about,?\s*[^.]*\.\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "No answer available"

test_answers_clean = [clean_generated(a) for a in test_answers]

def make_submission(test_df, predictions, filepath):
    sub = pd.DataFrame({
        'ID': test_df['ID'].values,
        'TargetRLF1': predictions, 'TargetR1F1': predictions, 'TargetLLM': predictions
    })
    assert list(sub.columns) == ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert len(sub) == len(test_df)
    assert sub['ID'].equals(test_df['ID'])
    for c in ['TargetRLF1', 'TargetR1F1', 'TargetLLM']:
        sub[c] = sub[c].replace('', 'No answer available').fillna('No answer available')
    assert sub.isnull().sum().sum() == 0
    sub.to_csv(filepath, index=False, encoding='utf-8')
    print(f"Saved {filepath} | {len(sub)} rows")
    return sub

sub = make_submission(test_df, test_answers_clean, '/kaggle/working/exp05_afriteva.csv')

# Show samples across languages for sanity check
print("\nSAMPLES:")
for subset in ['Aka_Gha', 'Amh_Eth', 'Swa_Ken', 'Eng_Uga']:
    idx = test_df[test_df['subset']==subset].index[0]
    pos = test_df.index.get_loc(idx)
    print(f"\n[{subset}] {test_df.loc[idx,'input'][:70]}")
    print(f"  -> {test_answers_clean[pos][:130]}")

Saved /kaggle/working/exp05_afriteva.csv | 2618 rows

SAMPLES:

[Aka_Gha] Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo 
  -> Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a ɛreyɛ adwuma de asiw GBV ano ma. Nkitahodi a Wɔde Di D

[Amh_Eth] ክላሚዲያ ሳይታከም ቢቆይ በወንዶች ላይ የሚያስከትለው የረጅም ጊዜ ጉዳት ምንድን ነው?
  -> ክላሚዲያ በወንዶች ላይ የአጭር ጊዜ ጉዳት ሊያስከትል ይችላል፣ ነገር ግን በእርግዝና ወቅት ሊቆይ ይችላል።

[Swa_Ken] Je, mtu mwenye afya nzuri anaweza kuambukizwa virusi vya ukimwi/UKIMWI
  -> Kuambukizwa virusi vya ukimwi/UKIMWI (Virusi vya Upungufu wa Kinga Mwilini) kunaweza kusababishwa na mambo mbalimbali, ikiwa ni pa

[Eng_Uga] Treatment for Gonorrhea?, please answer this using simple medical term
  -> Gonorrhea is treated with antibiotic tablets such as azithromycin or doxycycline. Your provider takes a sample of urine or secreti
